# Phân loại cảm xúc IMDb — hardcore

Model scratch **chưa train**; dự đoán chỉ minh họa inference, không phải kết quả chất lượng. Bạn tự thiết kế tiền xử lý, thay model và viết training sau khi đọc luồng vào/ra này.

## 1. Load dữ liệu cơ bản

Tập inference: **test có sẵn của dataset**. Nguồn/revision trong SOURCE/DATASET_ID. Dữ liệu thô được giữ nguyên; thao tác nhóm/sắp ID chỉ để giữ giao thức mẫu của hai bản đầy đủ, không fit đặc trưng.

Chạy từ thư mục dự án, sửa DATA_DIR trước Run All. DATA_MODE="native" đọc nguồn đã khai báo; DATA_MODE="local_jsonl" đọc bản xuất **đúng nguồn**, schema: test.jsonl: text:str, label:int theo LABEL_NAMES; giữ nguyên thứ tự canonical/ID của hai bản đầy đủ. Đường dẫn ảnh trong JSONL phải hợp lệ từ thư mục chạy. Nếu dùng nguồn HF cần cài datasets; COCO cần torchvision/pycocotools; TFDS cần kernel TensorFlow tương thích. Không tự thay dataset khi thiếu dữ liệu.

In [ ]:
from pathlib import Path
import hashlib
import io
import json
import random
import string
import unicodedata
import numpy as np
import torch
from torch import nn
from PIL import Image

SEED=42
BATCH_SIZE=16
MAX_INPUT_BYTES=4096  # Giới hạn đầu vào cố định của model; không đổi số mẫu inference.
MAX_OUTPUT_TOKENS=32
IMAGE_SIZE=64
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_MODE="native"  # Chọn local_jsonl rõ ràng để dùng bản xuất đúng dataset/schema bên dưới.

DATA_DIR=Path('data/sentiment_imdb_reviews')
CACHE_DIR=Path("artifacts/sentiment_imdb_reviews/hardcore/data_cache")
INFERENCE_SPLIT='test'
SOURCE='https://www.tensorflow.org/datasets/catalog/imdb_reviews'
DATASET_ID=None
DATASET_REVISION=None
TASK='imdb_reviews'
LABEL_NAMES=['neg', 'pos']


def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu {path}. Cung cấp JSONL từ đúng dataset/revision/schema đã ghi; không dùng dữ liệu giả thay thế.")
    with path.open(encoding="utf-8") as handle:
        rows=[json.loads(line) for line in handle if line.strip()]
    if not rows:raise ValueError(f"Tập dữ liệu rỗng: {path}")
    return rows

In [ ]:
def read_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def read_canonical_rows(rows, split):
    result = []
    for row in rows:
        if not isinstance(row.get("text"), str):
            raise ValueError("Schema cần text dạng chuỗi và label số nguyên.")
        y = row.get("label")
        if isinstance(y, bool) or not isinstance(y, int) or not 0 <= y < len(LABEL_NAMES):
            raise ValueError(f"Nhãn sai: {y!r}")
        text = row["text"]
        result.append({"text": text, "label": y, "key": read_hash(text + "\0" + str(y))})
    # Sắp theo nội dung để TFDS/JSONL/tar cùng nguồn có thứ tự chia giống nhau.
    result.sort(key=lambda x: (x["key"], x["text"]))
    return [{**r, "id": f"{split}:{i}:{r['key']}"} for i, r in enumerate(result)]

def load_raw(split):
    local_path = DATA_DIR / f"{split}.jsonl"
    if local_path.exists():
        print(f"Đọc dữ liệu gốc cục bộ: {local_path}")
        return read_canonical_rows(read_jsonl(local_path), split)
    if TASK == "imdb_reviews":
        archive = DATA_DIR / "aclImdb_v1.tar.gz"
        if archive.exists():
            import tarfile
            rows = []
            with tarfile.open(archive, "r:gz") as tar:
                for member in tar:
                    parts = member.name.split("/")
                    if (member.isfile() and len(parts) == 4 and parts[0] == "aclImdb"
                            and parts[1] == split and parts[2] in ("neg", "pos")
                            and parts[3].endswith(".txt")):
                        with tar.extractfile(member) as f:
                            # Cùng quy tắc với builder TFDS: đọc trọn review, strip hai đầu.
                            text = f.read().decode("utf-8").strip()
                        rows.append({"text": text, "label": int(parts[2] == "pos")})
            if not rows:
                raise ValueError(f"Không tìm thấy aclImdb/{split}/neg|pos/*.txt trong {archive}")
            print(f"Đọc archive Stanford gốc, cùng nguồn TFDS: {archive}")
            return read_canonical_rows(rows, split)
        try:
            import tensorflow as tf
            tf.config.set_visible_devices([], "GPU")  # Chỉ dùng TensorFlow để đọc dữ liệu CPU.
            import tensorflow_datasets as tfds
        except ImportError as exc:
            raise RuntimeError("Cài tensorflow tensorflow-datasets trong Python được hỗ trợ; "
                "hoặc đặt archive Stanford aclImdb_v1.tar.gz trong DATA_DIR. "
                "Có thể dùng train.jsonl/test.jsonl export đúng IMDb, text:str, label:0|1.") from exc
        try:
            ds, info = tfds.load("imdb_reviews/plain_text:1.0.0", split=split,
                as_supervised=True, with_info=True, shuffle_files=False,
                data_dir=str(DATA_DIR / "tfds"))
            assert info.features["label"].names == ["neg", "pos"]
            rows = [{"text": t.decode("utf-8"), "label": int(y)} for t,y in tfds.as_numpy(ds)]
        except Exception as exc:
            raise RuntimeError("Không tải/đọc được IMDb từ TFDS. Kiểm tra mạng, thư mục cache "
                "hoặc đặt archive Stanford gốc trong DATA_DIR; không đổi dataset.") from exc
    else:
        from sklearn.datasets import fetch_20newsgroups
        try:
            data = fetch_20newsgroups(subset=split, remove=("headers", "footers", "quotes"),
                shuffle=False, data_home=str(DATA_DIR / "sklearn"), download_if_missing=True)
        except Exception as exc:
            raise RuntimeError("Không tải/đọc được 20 Newsgroups. Kiểm tra mạng/cache; "
                "hoặc cung cấp train.jsonl/test.jsonl đã remove headers, footers, quotes, "
                "nhãn theo LABEL_NAMES. Không dùng fetch_20newsgroups_vectorized.") from exc
        assert list(data.target_names) == LABEL_NAMES, "Thứ tự nhãn khác khai báo."
        rows = [{"text": text, "label": int(y)} for text,y in zip(data.data, data.target)]
    return read_canonical_rows(rows, split)

records=load_raw("test")

assert records, "Không có mẫu test/validation để inference."
assert len({row["id"] for row in records})==len(records), "ID đầu vào bị trùng."
print("Tập inference:",INFERENCE_SPLIT,"; số mẫu:",len(records))

## 2. Định nghĩa một model và inference trực tiếp

Các lớp học đều khởi tạo ngẫu nhiên. Chỉ chuyển đầu vào tối thiểu bên trong model; không có tokenizer/vocabulary fit, hàm tiền xử lý riêng, loss, optimizer, training hay checkpoint. Giới hạn chiều đầu vào/đầu ra ở phần cấu hình là cố định và chỉ phục vụ model minh họa; toàn bộ mẫu của split vẫn được inference. Không truyền gold target vào forward.

In [ ]:
class RawClassifier(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.embedding=nn.Embedding(257,16)
        self.head=nn.Linear(16,num_classes)
    def forward(self,texts):
        outputs=[]
        for text in texts:
            ids=list(str(text).encode("utf-8")[:MAX_INPUT_BYTES]) or [256]
            values=torch.tensor(ids,device=self.embedding.weight.device)
            outputs.append(self.head(self.embedding(values).mean(0)))
        return torch.stack(outputs)

In [ ]:
model=RawClassifier(len(LABEL_NAMES)).to(DEVICE)
model.eval()
predictions=[]
with torch.inference_mode():
    for start in range(0,len(records),BATCH_SIZE):
        batch=records[start:start+BATCH_SIZE]
        inputs=[row["text"] for row in batch]
        outputs=model(inputs).argmax(-1).cpu().tolist()
        assert len(outputs)==len(batch), "Số dự đoán trong batch không khớp."
        predictions.extend([{'id':row["id"],'prediction':output} for row,output in zip(batch,outputs)])

assert len(predictions)==len(records)>0
assert [row['id'] for row in predictions]==[row["id"] for row in records]
assert len({row['id'] for row in predictions})==len(records)
assert all(isinstance(row['prediction'],int) and 0<=row['prediction']<len(LABEL_NAMES) for row in predictions)

print(f"Đã inference {len(predictions)} mẫu trên {INFERENCE_SPLIT} bằng model scratch CHƯA TRAIN.")
print("Đầu ra minh họa thực thi, chưa đo chất lượng; cần tự xây dựng tiền xử lý, model và training.")
print(predictions[:2])
# File dự đoán chỉ được tạo khi người dùng chạy notebook; không cần checkpoint/helper.
PREDICTION_PATH=Path("artifacts/sentiment_imdb_reviews/hardcore/predictions.json")
PREDICTION_PATH.parent.mkdir(parents=True,exist_ok=True)
PREDICTION_PATH.write_text(json.dumps(predictions,ensure_ascii=False,indent=2),encoding="utf-8")